# Notebook 04 — Interactive Visualizations

Produces self-contained HTML figures (no backend required).

Outputs:
- `figures/interactive/map_choropleth.html`
- `figures/interactive/scatter_explorer.html`
- `figures/interactive/linked_major_trend.html`

**All outputs must work without a server — open directly in a browser.**

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import json
from pathlib import Path

PROC = Path('../data/processed')
FIG  = Path('../figures/interactive')
FIG.mkdir(exist_ok=True)

# Project colors
COLORS = {
    'primary':   '#1A3A5C',
    'accent':    '#C0392B',
    'secondary': '#2980B9',
}

print('Plotly ready. Load processed data to begin.')

## Interactive #1 — Choropleth Map with Year Slider

In [ ]:
# state_year = pd.read_csv(PROC / 'state_level_timeseries.csv')
# Requires columns: state (2-letter abbr), year, korean_students

# fig = px.choropleth(
#     state_year,
#     locations='state',
#     locationmode='USA-states',
#     color='korean_students',
#     animation_frame='year',
#     scope='usa',
#     color_continuous_scale=[
#         [0.0, '#EBF5FB'], [0.25, '#AED6F1'],
#         [0.5, '#5DADE2'], [0.75, '#2E86C1'], [1.0, '#1A5276']
#     ],
#     labels={'korean_students': 'Korean Students'},
#     title='Korean Students in U.S. by State and Year',
#     hover_name='state',
# )
# fig.update_layout(
#     geo=dict(bgcolor='rgba(0,0,0,0)'),
#     paper_bgcolor='white',
#     font_family='Arial',
# )
# fig.write_html(
#     FIG / 'map_choropleth.html',
#     include_plotlyjs='cdn',   # keeps file size small
#     full_html=True,
# )
# print('Saved map_choropleth.html')
print('Choropleth — implement after state_level_timeseries.csv is ready.')

## Interactive #2 — Multi-Variable Scatter Plot

In [ ]:
# master = pd.read_csv(PROC / 'korean_students_master.csv')
# latest_year = master.groupby('institution_name').apply(lambda g: g[g['year'] == g['year'].max()]).reset_index(drop=True)

# Create dropdown for X axis variable selection
# Variables: qs_rank, tuition_out_of_state, stem_opt_programs, state_korean_pop

# fig = go.Figure()
# ... build with updatemenus for axis switching

# fig.write_html(
#     FIG / 'scatter_explorer.html',
#     include_plotlyjs='cdn',
#     full_html=True,
# )
# print('Saved scatter_explorer.html')
print('Scatter explorer — implement after master dataset is ready.')

## Linked View — Field of Study Donut → Enrollment Trend

This linked view is built in D3.js (written as a self-contained HTML file).

In [ ]:
# The linked view is a D3.js visualization written to an HTML file.
# It does not use Plotly — see the template below.

fields_path = PROC / 'field_of_study_timeseries.csv'

if fields_path.exists():
    fields_df = pd.read_csv(fields_path)
    # Convert to JSON for embedding in HTML
    fields_json = fields_df.to_json(orient='records')
    print(f'Loaded {len(fields_df)} rows; embedding in HTML...')
    # TODO: embed fields_json into the D3 HTML template below and save to FIG
else:
    print('⚠️  Run data cleaning notebook first to generate field_of_study_timeseries.csv')

### D3.js Linked View Template

The cell below writes the self-contained HTML/D3 file.
Fill in the `DATA_PLACEHOLDER` with the JSON from the cell above.

In [ ]:
LINKED_HTML_TEMPLATE = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Major Field vs Enrollment — Linked View</title>
  <script src="https://d3js.org/d3.v7.min.js"></script>
  <style>
    body { font-family: Arial, sans-serif; margin: 0; background: #fff; }
    #container { display: flex; gap: 2rem; padding: 1.5rem; }
    #donut-panel { width: 320px; flex-shrink: 0; }
    #line-panel  { flex: 1; }
    h3 { color: #1A3A5C; font-size: 1rem; margin-bottom: 0.5rem; }
    .arc path { cursor: pointer; stroke: white; stroke-width: 1.5px; transition: opacity 0.2s; }
    .arc path:hover { opacity: 0.8; }
    .arc.selected path { stroke: #1A3A5C; stroke-width: 3px; }
    .legend-item { cursor: pointer; font-size: 0.82rem; }
    .annotation-line { stroke: #C0392B; stroke-dasharray: 4; stroke-width: 1.5; }
    .caption { font-size: 0.72rem; color: #7F8C8D; margin-top: 0.5rem; }
  </style>
</head>
<body>
<div id="container">
  <div id="donut-panel">
    <h3>Click a field to explore its trend →</h3>
    <svg id="donut"></svg>
    <div id="legend"></div>
  </div>
  <div id="line-panel">
    <h3 id="line-title">Select a field of study</h3>
    <svg id="linechart"></svg>
    <p class="caption">Source: IIE Open Doors, Fields of Study by Place of Origin.</p>
  </div>
</div>
<script>
  const RAW_DATA = DATA_PLACEHOLDER;

  const FIELD_COLORS = {
    "Business":             "#1A3A5C",
    "STEM":                 "#2980B9",
    "Humanities & Social": "#27AE60",
    "Other":               "#F39C12"
  };

  // Parse and group
  const byField = d3.group(RAW_DATA, d => d.field_category);
  const latestYear = d3.max(RAW_DATA, d => +d.year);
  const latestData = RAW_DATA.filter(d => +d.year === latestYear);

  // ── Donut chart ─────────────────────────────────────────────────
  const donutW = 280, donutH = 260, radius = Math.min(donutW, donutH) / 2 - 10;
  const svgDonut = d3.select("#donut").attr("width", donutW).attr("height", donutH)
    .append("g").attr("transform", `translate(${donutW/2},${donutH/2})`);

  const pie = d3.pie().value(d => d.korean_students).sort(null);
  const arc = d3.arc().innerRadius(radius*0.55).outerRadius(radius);

  let selectedField = null;

  function drawDonut(field = null) {
    svgDonut.selectAll(".arc").remove();
    const data = latestData;
    svgDonut.selectAll(".arc").data(pie(data)).enter()
      .append("g").attr("class", d => `arc${field === d.data.field_category ? " selected" : ""}`)
      .append("path")
      .attr("d", arc)
      .attr("fill", d => FIELD_COLORS[d.data.field_category] || "#ccc")
      .on("click", (event, d) => {
        selectedField = d.data.field_category;
        drawDonut(selectedField);
        updateLine(selectedField);
      });
  }
  drawDonut();

  // ── Line chart ───────────────────────────────────────────────────
  const lW = 550, lH = 320;
  const margin = {top: 20, right: 30, bottom: 40, left: 60};
  const lIW = lW - margin.left - margin.right;
  const lIH = lH - margin.top - margin.bottom;

  const svgLine = d3.select("#linechart").attr("width", lW).attr("height", lH)
    .append("g").attr("transform", `translate(${margin.left},${margin.top})`);

  const xScale = d3.scaleLinear().range([0, lIW])
    .domain(d3.extent(RAW_DATA, d => +d.year));
  const yScale = d3.scaleLinear().range([lIH, 0]);

  svgLine.append("g").attr("class","x-axis").attr("transform",`translate(0,${lIH})`);
  svgLine.append("g").attr("class","y-axis");
  svgLine.append("path").attr("class","trend-line")
    .attr("fill","none").attr("stroke-width", 2.5);
  svgLine.append("line").attr("class","annotation-line")
    .attr("x1", xScale(2016)).attr("x2", xScale(2016))
    .attr("y1", 0).attr("y2", lIH);
  svgLine.append("text")
    .attr("x", xScale(2016)+4).attr("y", 12)
    .text("STEM OPT expansion").style("font-size","0.7rem").style("fill","#C0392B");

  function updateLine(field) {
    d3.select("#line-title").text(`Korean Students — ${field}`);
    const data = (byField.get(field) || []).sort((a,b) => a.year - b.year);
    if (!data.length) return;
    yScale.domain([0, d3.max(data, d => +d.korean_students) * 1.1]);
    svgLine.select(".x-axis").call(d3.axisBottom(xScale).tickFormat(d3.format("d")));
    svgLine.select(".y-axis").call(d3.axisLeft(yScale).ticks(5)
      .tickFormat(d => d >= 1000 ? `${d/1000}k` : d));
    const line = d3.line().x(d => xScale(+d.year)).y(d => yScale(+d.korean_students)).curve(d3.curveMonotoneX);
    svgLine.select(".trend-line")
      .datum(data).transition().duration(300)
      .attr("d", line).attr("stroke", FIELD_COLORS[field] || "#1A3A5C");
    svgLine.select(".annotation-line").attr("y2", lIH);
  }
</script>
</body>
</html>'''

# To use: replace DATA_PLACEHOLDER with the JSON string from fields_df
# then write to file:
# html_out = LINKED_HTML_TEMPLATE.replace('DATA_PLACEHOLDER', fields_json)
# with open(FIG / 'linked_major_trend.html', 'w') as f:
#     f.write(html_out)
# print('Saved linked_major_trend.html')
print('Linked view template ready. Replace DATA_PLACEHOLDER with fields_json once data is available.')